<a href="https://colab.research.google.com/github/Sampaioooo/fake-news-bertimbau-tde/blob/main/notebooks/fake_news_bertimbau.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U "transformers>=4.44.0" "datasets==3.6.0" "accelerate>=0.33.0" evaluate scikit-learn matplotlib
!pip -q install "pandas==2.2.2"

In [ ]:
import pandas as pd

print(pd.__version__)

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo usado:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Atenção: a GPU não está ativa. O treinamento pode ficar lento.")

In [ ]:
dataset = load_dataset(
    "fake-news-UFG/fakebr",
    "full_texts",
    trust_remote_code=True
)

dataset

In [ ]:
dados = dataset["train"]

print("Quantidade total de notícias:", len(dados))
print("Colunas disponíveis:", dados.column_names)
print("Classes:", dados.features["label"].names)

exemplo = dados[0]

print("\nClasse do primeiro exemplo:", dados.features["label"].int2str(exemplo["label"]))
print("\nTexto do primeiro exemplo:")
print(exemplo["text"][:1000])

In [ ]:
dados_limpos = dados.select_columns(["text", "label"])

print(dados_limpos)
print("Classes:", dados_limpos.features["label"].names)
print("Exemplo:", dados_limpos[0])

In [ ]:
from datasets import config

config.TORCHVISION_AVAILABLE = False

print("Torchvision desativado para evitar conflito com datasets.")

In [ ]:
divisao_inicial = dados_limpos.train_test_split(
    test_size=0.30,
    seed=42,
    stratify_by_column="label"
)

divisao_validacao_teste = divisao_inicial["test"].train_test_split(
    test_size=0.50,
    seed=42,
    stratify_by_column="label"
)

dataset_final = DatasetDict({
    "train": divisao_inicial["train"],
    "validation": divisao_validacao_teste["train"],
    "test": divisao_validacao_teste["test"]
})

dataset_final

In [ ]:
for parte in dataset_final:
    labels = dataset_final[parte]["label"]
    contagem = pd.Series(labels).value_counts().sort_index()

    print(f"\n{parte.upper()}")
    for indice, quantidade in contagem.items():
        nome_classe = dados_limpos.features["label"].int2str(indice)
        print(f"{nome_classe}: {quantidade}")

In [ ]:
modelo_base = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)

print("Tokenizer carregado:", modelo_base)

In [ ]:
def tokenizar(exemplos):
    return tokenizer(
        exemplos["text"],
        truncation=True,
        max_length=256
    )

dataset_tokenizado = dataset_final.map(
    tokenizar,
    batched=True,
    remove_columns=["text"]
)

dataset_tokenizado

In [ ]:
id2label = {
    0: "fake",
    1: "true"
}

label2id = {
    "fake": 0,
    "true": 1
}

modelo = AutoModelForSequenceClassification.from_pretrained(
    modelo_base,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def calcular_metricas(pred):
    logits, labels = pred
    predicoes = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predicoes)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predicoes,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Modelo carregado com sucesso.")

In [ ]:
argumentos_treinamento = TrainingArguments(
    output_dir="./resultados_bertimbau_fake_news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none"
)

try:
    trainer = Trainer(
        model=modelo,
        args=argumentos_treinamento,
        train_dataset=dataset_tokenizado["train"],
        eval_dataset=dataset_tokenizado["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=calcular_metricas
    )

except TypeError:
    trainer = Trainer(
        model=modelo,
        args=argumentos_treinamento,
        train_dataset=dataset_tokenizado["train"],
        eval_dataset=dataset_tokenizado["validation"],
        data_collator=data_collator,
        compute_metrics=calcular_metricas
    )

print("Treinamento configurado.")

In [ ]:
trainer.train()

In [ ]:
resultado_teste = trainer.evaluate(dataset_tokenizado["test"])

resultado_teste

In [ ]:
predicoes = trainer.predict(dataset_tokenizado["test"])

y_real = predicoes.label_ids
y_pred = np.argmax(predicoes.predictions, axis=1)

nomes_classes = ["fake", "true"]

print(classification_report(
    y_real,
    y_pred,
    target_names=nomes_classes,
    digits=4
))

In [ ]:
matriz = confusion_matrix(y_real, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=nomes_classes
)

disp.plot(values_format="d")
plt.title("Matriz de Confusão - BERTimbau Fake News")
plt.show()

In [ ]:
def prever_noticia(texto):
    entradas = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    entradas = {chave: valor.to(modelo.device) for chave, valor in entradas.items()}

    with torch.no_grad():
        saida = modelo(**entradas)
        probabilidades = torch.softmax(saida.logits, dim=1)
        classe_prevista = torch.argmax(probabilidades, dim=1).item()

    return {
        "classe": id2label[classe_prevista],
        "probabilidade_fake": float(probabilidades[0][0]),
        "probabilidade_true": float(probabilidades[0][1])
    }

texto_teste_1 = "Pesquisadores anunciam nova tecnologia para melhorar a detecção de doenças em exames médicos."
texto_teste_2 = "Urgente! Governo confirma que todos os bancos vão bloquear contas amanhã sem aviso."

print(prever_noticia(texto_teste_1))
print(prever_noticia(texto_teste_2))

In [ ]:
import os

os.makedirs("resultados", exist_ok=True)
os.makedirs("modelo_treinado", exist_ok=True)

print("Pastas criadas com sucesso.")

In [ ]:
metricas_teste = {
    "accuracy": resultado_teste.get("eval_accuracy"),
    "precision": resultado_teste.get("eval_precision"),
    "recall": resultado_teste.get("eval_recall"),
    "f1": resultado_teste.get("eval_f1"),
    "loss": resultado_teste.get("eval_loss")
}

df_metricas = pd.DataFrame([metricas_teste])

df_metricas.to_csv("resultados/metricas_teste.csv", index=False)

df_metricas

In [ ]:
relatorio_dict = classification_report(
    y_real,
    y_pred,
    target_names=nomes_classes,
    digits=4,
    output_dict=True
)

df_relatorio = pd.DataFrame(relatorio_dict).transpose()

df_relatorio.to_csv("resultados/relatorio_classificacao.csv")

df_relatorio

In [ ]:
matriz = confusion_matrix(y_real, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=nomes_classes
)

disp.plot(values_format="d")
plt.title("Matriz de Confusão - BERTimbau Fake News")
plt.savefig("resultados/matriz_confusao.png", dpi=300, bbox_inches="tight")
plt.show()

print("Matriz de confusão salva em: resultados/matriz_confusao.png")

In [ ]:
modelo.save_pretrained("modelo_treinado")
tokenizer.save_pretrained("modelo_treinado")

print("Modelo e tokenizer salvos na pasta modelo_treinado.")

In [ ]:
textos_teste = [
    "Pesquisadores anunciam nova tecnologia para melhorar a detecção de doenças em exames médicos.",
    "Urgente! Governo confirma que todos os bancos vão bloquear contas amanhã sem aviso.",
    "Universidade divulga calendário acadêmico atualizado para o próximo semestre.",
    "Cientistas revelam que beber água com limão cura todas as doenças em 24 horas."
]

resultados_manuais = []

for texto in textos_teste:
    resultado = prever_noticia(texto)

    resultados_manuais.append({
        "texto": texto,
        "classe_prevista": resultado["classe"],
        "probabilidade_fake": resultado["probabilidade_fake"],
        "probabilidade_true": resultado["probabilidade_true"]
    })

df_testes_manuais = pd.DataFrame(resultados_manuais)

df_testes_manuais.to_csv("resultados/testes_manuais.csv", index=False)

df_testes_manuais

In [ ]:
!zip -r resultados_projeto_fake_news.zip resultados modelo_treinado

In [ ]:
print("CONCLUSÃO TÉCNICA DO EXPERIMENTO")
print("-" * 50)
print("O modelo BERTimbau foi ajustado para classificar notícias em português como fake ou true.")
print("A base Fake.Br foi utilizada por conter notícias falsas e verdadeiras em português brasileiro.")
print("No conjunto de teste, o modelo alcançou aproximadamente 99,63% de acurácia, precisão, recall e F1-score.")
print("A matriz de confusão indicou apenas 4 erros em 1080 exemplos testados.")
print("Apesar do bom desempenho, os testes manuais com frases curtas mostraram que o modelo pode ter dificuldade fora do padrão da base original.")
print("Portanto, o modelo apresenta bom desempenho experimental, mas ainda precisaria de validação com novos dados antes de uso em ambiente real.")